In [ ]:
# 경주님 함수 적용 X (train 결측치 제거 컬럼 149개)

import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)

# 데이터 확인하기 2025.11.20 152개 컬럼을 사용해 RandomForest 기본 모델 돌리기
import pandas as pd
import numpy  as np

from sklearn.preprocessing import StandardScaler # 데이터 전처리용

import matplotlib.pyplot as plt
import seaborn as sns

from utils.preprocessing import load_data, split_features_target
from utils.model_utils   import save_model
from utils.user_utils    import get_clf_eval

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")

# import importlib
# importlib.reload(preprocessing)


In [3]:
# 데이터 로딩
data_path_train = '../data/train.csv'
data_path_test  = '../data/test.csv'

train = pd.read_csv(data_path_train)
test  = pd.read_csv(data_path_test)

In [4]:
# Data 전처리 1. zero_count_rate이 99%인 컬럼 제거하기 
remove_cols = pd.read_csv('../doc/remove_cols.xls', header=0).squeeze() 
train.drop(columns=remove_cols, axis=1, inplace=True)
test.drop(columns=remove_cols, axis=1, inplace=True)

In [5]:
# 데이터 할당
X_features = train.drop(columns=['ID', 'TARGET'], axis=1) # ID와 TARGET 모두 제거하고 X_train 만들기
y_labels   = train['TARGET']
X_test     = test.drop(columns=['ID'], axis=1) # test 데이터에서도 ID 제거 
X_features.columns # 149개

Index(['var3', 'var15', 'imp_ent_var16_ult1', 'imp_op_var39_comer_ult1',
       'imp_op_var39_comer_ult3', 'imp_op_var41_comer_ult1',
       'imp_op_var41_comer_ult3', 'imp_op_var41_efect_ult1',
       'imp_op_var41_efect_ult3', 'imp_op_var41_ult1',
       ...
       'saldo_medio_var8_ult3', 'saldo_medio_var12_hace2',
       'saldo_medio_var12_hace3', 'saldo_medio_var12_ult1',
       'saldo_medio_var12_ult3', 'saldo_medio_var13_corto_hace2',
       'saldo_medio_var13_corto_hace3', 'saldo_medio_var13_corto_ult1',
       'saldo_medio_var13_corto_ult3', 'var38'],
      dtype='object', length=149)

In [6]:
# Data 전처리 2. var3 의 최소값 -99999 를 최빈값으로 변경하기
X_features['var3'] = X_features['var3'].replace(-999999, 2)

In [7]:
# 스케일링
# X_train_scaled, X_test_scaled, scaler = scale_data(X_train=X, X_test=X_test)
scaler        = StandardScaler()
X_scaled      = scaler.fit_transform(X_features)
X_test_scaled = scaler.transform(X_test)

In [8]:
# 레이블의 분포 확인
cust_cnt = y_labels.value_counts()
print(cust_cnt) # 1이 불만족 3008명, 만족이 73012

# 불만족고객의 비율
cust_rate = cust_cnt[1] / cust_cnt.sum()
print(f'불만족 고객 비율: {cust_rate:.2f}')

TARGET
0    73012
1     3008
Name: count, dtype: int64
불만족 고객 비율: 0.04


In [9]:
type(X_scaled)

numpy.ndarray

In [10]:
# first testing model
# XGBoost (xgb) : yjh, kjh
# LightGBM(lgbm) : lsj, ujm
# Random Forest(rf) : lkj, kjh
# Logistic Regression(lr) : yjh, ujm


In [11]:
# 학습/테스트 데이터 분리
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
  X_features, 
  y_labels,
  test_size    = 0.2, # 8:2
  random_state = 23, # 세미프로젝트3조
  stratify = y_labels
)


In [11]:
# Logistic Regression 

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# 모델 생성 및 학습
log_reg = LogisticRegression(max_iter=1000, random_state=23)
log_reg.fit(X_train, y_train)

# 예측
y_pred = log_reg.predict(X_val)
pred_proba = log_reg.predict_proba(X_val)[:,1]

# 평가
print("Accuracy:", accuracy_score(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, pred_proba))

# Accuracy: 0.9604051565377533
# ROC-AUC: 0.6228827480511704

Accuracy: 0.9604051565377533
ROC-AUC: 0.6228827480511704


In [12]:
# XGBoost

from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=23
)

xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_val)
pred_proba = xgb.predict_proba(X_val)[:,1]

print("Accuracy:", accuracy_score(y_val, xgb_pred))
print("ROC-AUC:", roc_auc_score(y_val, pred_proba))

# Accuracy: 0.9606024730334123
# ROC-AUC: 0.841652897864535

Accuracy: 0.9606024730334123
ROC-AUC: 0.841652897864535


In [ ]:
# 경주님 만들어놓은 함수 불러와서 학습
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# 기본값으로 우선 LogisticRegression 
log_reg_clf = LogisticRegression(max_iter=1000, random_state=23)
log_reg_clf.fit(X_train, y_train)    # 학습
save_model(log_reg_clf, 'LogisticRegression_basic')
pred       = log_reg_clf.predict(X_val)   # 예측
pred_proba = log_reg_clf.predict_proba(X_val)[:,1] # 예측확률

get_clf_eval(y_test=y_val, pred=pred, pred_proba=pred_proba)

'''
warning 발생
lbfgs failed to converge after 1000 iteration(s)
- 원인: 로지스틱 회귀의 최적화 알고리즘(lbfgs)이 1000번 반복해도 수렴하지 못한 경우

- 반복 횟수 늘리기
log_reg = LogisticRegression(max_iter=5000, random_state=23)

- 데이터 스케일링
StandardScaler나 MinMaxScaler로 입력 데이터를 정규화하면 수렴이 잘 됩니다
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

- 다른 solver 사용
예: solver="saga" 또는 solver="liblinear"
log_reg = LogisticRegression(max_iter=2000, solver="saga", random_state=23)


UndefinedMetricWarning (Precision = 0.0
Precision is ill-defined and being set to 0.0 due to no predicted samples
- 원인: 모델이 검증 데이터에서 **양성 클래스(예: 1)**를 전혀 예측하지 않았을 때 발생합니다.

- zero_division 옵션으로 경고를 제어
from sklearn.metrics import precision_score
precision_score(y_val, y_pred, zero_division=0)
- 하지만 근본적으로는 모델이 양성 클래스를 전혀 못 맞추고 있다는 것이 문제입니다.
→ 데이터 불균형이 심하거나, 모델이 너무 단순해서 양성 클래스 학습을 못했을 가능성이 큽니다.

- 클래스 불균형 처리: class_weight="balanced" 옵션 추가
log_reg = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=23)
- 다른 모델 시도: RandomForest, XGBoost 등 트리 기반 모델은 불균형 데이터에 더 강합니다.
'''

✓ 모델 저장 완료: models\LogisticRegression_basic.pkl
  파일 크기: 0.00 MB
AUC: 0.6229, 정확도: 0.9604, 정밀도: 0.0000, 재현율: 0.0000, F1: 0.0000
오차행렬:
[[14602     0]
 [  602     0]]


c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [13]:
from xgboost import XGBClassifier

# 기본값으로 우선 XGBoost
xgb_clf = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=23
)
xgb_clf.fit(X_train, y_train)    # 학습
save_model(xgb_clf, 'XGBoost_basic')
pred       = xgb_clf.predict(X_val)   # 예측
pred_proba = xgb_clf.predict_proba(X_val)[:,1] # 예측확률

get_clf_eval(y_test=y_val, pred=pred, pred_proba=pred_proba)

✓ 모델 저장 완료: models\XGBoost_basic.pkl
  파일 크기: 1.46 MB
AUC: 0.8417, 정확도: 0.9606, 정밀도: 0.6667, 재현율: 0.0100, F1: 0.0196
오차행렬:
[[14599     3]
 [  596     6]]
